# Key-Query Analysis

Analyze how keys and queries interact: alignment, subspace structure,
position dependence, match profiles, and QK weight decomposition.

## Why This Matters

QK circuit key query analyzes the query-key interaction that determines attention patterns. Understanding what features drive attention — positional vs. content-based, local vs. global — reveals how the model decides which information to route where.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.key_query_analysis import (
    key_query_alignment, key_query_subspace,
    key_query_position_dependence, key_query_match_profile,
    key_query_weight_decomposition,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Key-Query Alignment

How well do keys and queries align at each position? Self-alignment and cross-alignment.

In [ ]:
for layer in range(4):
    result = key_query_alignment(model, tokens, layer=layer)
    print(f'Layer {layer}:')
    for h in result['per_head']:
        print(f"  Head {h['head']}: self_align={h['mean_self_alignment']:+.4f}, "
              f"cross_align={h['mean_cross_alignment']:+.4f}, "
              f"q_norm={h['q_mean_norm']:.4f}, k_norm={h['k_mean_norm']:.4f}")
    print()

## Key-Query Subspace

Effective dimensionality of key and query spaces per head.

In [ ]:
for layer in range(4):
    result = key_query_subspace(model, tokens, layer=layer)
    print(f'Layer {layer}:')
    for h in result['per_head']:
        print(f"  Head {h['head']}: q_rank={h['q_effective_rank']:.2f} "
              f"(top_sv={h['q_top_sv_fraction']:.1%}), "
              f"k_rank={h['k_effective_rank']:.2f} "
              f"(top_sv={h['k_top_sv_fraction']:.1%})")
    print()

## Position Dependence

Are keys/queries driven by position or content?

In [ ]:
for layer in range(4):
    result = key_query_position_dependence(model, tokens, layer=layer)
    print(f"Layer {layer} ({result['n_position_dependent']} position-dependent):")
    for h in result['per_head']:
        dep = ' [POS-DEP]' if h['is_position_dependent'] else ''
        print(f"  Head {h['head']}: q_var={h['q_direction_variation']:.4f}, "
              f"k_var={h['k_direction_variation']:.4f}, "
              f"q_cv={h['q_norm_cv']:.4f}, k_cv={h['k_norm_cv']:.4f}{dep}")
    print()

## Match Profile

Which keys does a specific query match best? Pre-softmax scores and cosines.

In [ ]:
result = key_query_match_profile(model, tokens, layer=0, head=0, position=-1)
print(f"Query at pos {result['query_position']} (tok {result['query_token']}), "
      f"norm={result['query_norm']:.4f}\n")
for k in result['per_key']:
    bar = '█' * max(0, int((k['score'] + 1) * 10))
    print(f"  Key pos {k['key_position']} (tok {k['key_token']}): "
          f"score={k['score']:+.4f}, cos={k['cosine']:+.4f}, "
          f"k_norm={k['key_norm']:.4f} {bar}")

## QK Weight Decomposition

Analyze the W_Q^T W_K weight circuit: rank, symmetry, spectral structure.

In [ ]:
for layer in range(4):
    print(f'Layer {layer}:')
    for head in range(4):
        result = key_query_weight_decomposition(model, layer=layer, head=head)
        print(f"  Head {head}: spec_norm={result['spectral_norm']:.4f}, "
              f"eff_rank={result['effective_rank']:.2f}, "
              f"top_sv={result['top_sv_fraction']:.1%}, "
              f"symmetry={result['symmetry_fraction']:.1%}")
    print()